# Plotting Time Series

Lets-Plot handles all temporal data types through a unified "datetime" scale (excluding duration, which is handled by the "time" scale). <br>
This is in contrast to R's ggplot2, which provides separate "date", "time", and "datetime" scales.

**Supported temporal data types:**

**kotlinx.datetime** library:

- `Instant`
- `LocalDate`
- `LocalTime`
- `LocalDateTime`

**java.time** package:

- `Instant`
- `LocalDate`
- `LocalTime`
- `LocalDateTime`
- `ZonedDateTime` (timezone-aware)
- `OffsetDateTime` (timezone-aware with offset)

**java.util** package:

- `Date`


In [1]:
%useLatestDescriptors
%use lets-plot

In [2]:
LetsPlot.getInfo()

Lets-Plot Kotlin API v.4.11.0. Frontend: Notebook with dynamically loaded JS. Lets-Plot JS v.4.7.0.

In [3]:
import kotlin.math.*

fun squiggle(x: Double): Double {
    return sin(3 * x) / (x * (cos(x) + 2))
}

In [4]:
val N = 50
val xs = (0 until N).map { i -> 1.0 + (25.0 - 1.0) * i / (N - 1) }
val ys = xs.map { squiggle(it) }

#### Local Time

`kotlinx.datetime.LocalTime` and `java.time.LocalTime` represent time-of-day values (local/clock time) independent of any specific date. 

The `datetime` scale renders these with default tooltips and scale breaks optimized for hours or smaller time units.



In [5]:
import kotlinx.datetime.LocalTime

// Create time objects at regular intervals: every 12 minutes starting from 8:00 AM
val startTime = LocalTime(8, 0)  // 8:00 AM
val intervalMinutes = 12
val xsTime = (0 until N).map { i ->
    val totalMinutes = i * intervalMinutes
    val hours = 8 + totalMinutes / 60
    val minutes = totalMinutes % 60
    LocalTime(hours, minutes)
}

val data = mapOf(
    "xs" to xsTime,
    "ys" to ys
)

(letsPlot(data = data) { x = "xs"; y = "ys" } +
    geomBand(
        xmin = LocalTime(12, 30),        // <-- Use a compatible datatype ('LocalTime' here) for a constant value.
        xmax = LocalTime(13, 30),
        tooltips = layerTooltips()
            .title("Lunch Time")
            .line("^xmin - ^xmax")
            .line("in any time zone")
    ) +
    geomLine(
        tooltips = layerTooltips()
            .line("Time:|@xs")
            .format("^x", "%H:%M:%S"),
        color = "#1380A1", 
        size = 2
    ) +
    ggtb() + 
    ggsize(800, 400) + 
    theme(axisTitle = "blank")
)    

#### Local Date

`kotlinx.datetime.LocalDate` and `java.time.LocalDate` represent calendar dates without time information. 

`LocalDate` objects are inherently timezone-agnostic since they contain no time component - a date like "2024-01-15" represents the same calendar day regardless of timezone.

The `datetime` scale renders these with default tooltips and scale breaks optimized for days or larger time units (weeks, months, years) depending on the data range.

In [6]:
import kotlinx.datetime.LocalDate
import kotlinx.datetime.DatePeriod
import kotlinx.datetime.plus

// This covers March 15 - May 3, 2025
val startDate = LocalDate(2025, 3, 15)
val xsDate = (0 until N).map { i ->
    startDate + DatePeriod(days = i)
}

val data = mapOf(
    "xs" to xsDate,
    "ys" to ys
)

(letsPlot(data = data) { x = "xs"; y = "ys" } +
    geomVLine(
        xintercept = LocalDate(2025, 4, 1),
        tooltips = layerTooltips()
            .title("April Fools' Day")
            .format("^xintercept", "%a, %b %e, %Y"),
        linetype = "dotted", 
        size = 1.5
    ) +
    geomLine(
        tooltips = layerTooltips()
            .line("Date:|@xs")
            .format("^x", "%b %d, %Y"),
        color = "#1380A1", 
        size = 2
    ) +
    ggtb() + 
    ggsize(800, 400) + 
    theme(axisTitle = "blank")
)    

#### Date-Time

`kotlinx.datetime.LocalDateTime`, `java.time.ZonedDateTime`, and `java.time.OffsetDateTime` represent both date and time information, either timezone-naive or timezone-aware.

The datetime scale automatically adapts tooltips and scale breaks based on the data's temporal resolution.

**Timezone handling:** If no timezone information is present (`LocalDateTime`), Lets-Plot assumes UTC timezone. <br>
For timezone-aware datetime objects (`ZonedDateTime`/`OffsetDateTime`), the timezone information from the data is preserved and used for rendering.

In [7]:
import java.time.ZonedDateTime
import java.time.ZoneId
import java.time.Instant

// Berlin DST transition: March 30, 2025 at 2:00 AM
// Cover ±3 days: March 27 - April 2, 2025 (6 days total)

// Generate equally spaced UNIX timestamps (in seconds)
val startTimestamp = ZonedDateTime.of(2025, 3, 27, 0, 0, 0, 0, ZoneId.of("UTC")).toEpochSecond()
val endTimestamp = ZonedDateTime.of(2025, 4, 2, 23, 59, 0, 0, ZoneId.of("UTC")).toEpochSecond()
val unixTimestamps = (0 until N).map { i -> 
    startTimestamp + (endTimestamp - startTimestamp) * i / (N - 1) 
}

// Convert to ZonedDateTime with Berlin timezone
val xsDatetime = unixTimestamps.map { timestamp ->
    Instant.ofEpochSecond(timestamp).atZone(ZoneId.of("Europe/Berlin"))
}

val data = mapOf(
    "xs" to xsDatetime,
    "ys" to ys
)

val p = letsPlot(data = data) { x = "xs"; y = "ys" } +
    geomVLine(
        xintercept = ZonedDateTime.of(2025, 3, 30, 1, 59, 59, 0, ZoneId.of("Europe/Berlin")),
        tooltips = layerTooltips()
            .title("Berlin \"spring forward\" transition")
            .line("^xintercept").format("^xintercept", "%b %d, %Y %H:%M:%S")
            .line("2:00 AM → 3:00 AM"),
        linetype = "dotted", 
        size = 1.5
    ) +
    geomLine(
        tooltips = layerTooltips()
            .line("@xs")
            .format("^x", "%b %d, %Y %H:%M"),
        color = "#1380A1", 
        size = 2
    ) +
    ggtb() + 
    ggsize(800, 400) + 
    theme(axisTitle = "blank")

p    

#### Zoomed-in View: Berlin Start of DST Transition Details

This plot zooms into the DST transition period: ±8 hours around March 30, 2025 at 2:00 AM.

Notice that the time 2025-03-30 02:00:00 through 2025-03-30 02:59:59 does not exist in Europe/Berlin timezone - clocks spring forward directly from 01:59:59 to 03:00:00, creating a gap in the timeline.

In [8]:
p + coordCartesian(
    xlim = Pair(
        ZonedDateTime.of(2025, 3, 29, 21, 0, 0, 0, ZoneId.of("Europe/Berlin")),
        ZonedDateTime.of(2025, 3, 30, 8, 0, 0, 0, ZoneId.of("Europe/Berlin"))
    )
)

#### Berlin End of DST Transition Details

The mirroring phenomenon can be observed during Berlin's end of DST transition: around October 26, 2025 at 2:00 AM.

Notice that the time 2025-10-26 02:00:00 through 2025-10-26 02:59:59 appears twice in Europe/Berlin timezone - clocks fall back from 02:59:59 to 02:00:00, creating a duplication in the timeline where the same hour occurs twice.

In [9]:
// Berlin DST transition back: October 26, 2025 at 3:00 AM → 2:00 AM

// Generate equally spaced UNIX timestamps (in seconds)
val startTimestamp = ZonedDateTime.of(2025, 10, 23, 0, 0, 0, 0, ZoneId.of("UTC")).toEpochSecond()
val endTimestamp = ZonedDateTime.of(2025, 10, 29, 23, 59, 0, 0, ZoneId.of("UTC")).toEpochSecond()
val unixTimestamps = (0 until N).map { i -> 
    startTimestamp + (endTimestamp - startTimestamp) * i / (N - 1) 
}

// Convert to ZonedDateTime with Berlin timezone
val xsDatetime = unixTimestamps.map { timestamp ->
    Instant.ofEpochSecond(timestamp).atZone(ZoneId.of("Europe/Berlin"))
}

val data = mapOf(
    "xs" to xsDatetime,
    "ys" to ys
)

val p = letsPlot(data = data) { x = "xs"; y = "ys" } +
    geomVLine(
        xintercept = Instant.ofEpochSecond(
            ZonedDateTime.of(2025, 10, 26, 0, 59, 59, 0, ZoneId.of("UTC")).toEpochSecond()
        ).atZone(ZoneId.of("Europe/Berlin")),
        tooltips = layerTooltips()
            .title("Berlin \"fall back\" transition")
            .line("^xintercept").format("^xintercept", "%b %d, %Y %H:%M:%S")
            .line("3:00 AM → 2:00 AM"),
        linetype = "dotted", 
        size = 1.5
    ) +
    geomLine(
        tooltips = layerTooltips()
            .line("@xs")
            .format("^x", "%b %d, %Y %H:%M"),
        color = "#1380A1", 
        size = 2
    ) +
    ggtb() + 
    ggsize(800, 400) + 
    theme(axisTitle = "blank")

p + coordCartesian(
    xlim = Pair(
        ZonedDateTime.of(2025, 10, 25, 22, 0, 0, 0, ZoneId.of("Europe/Berlin")),
        ZonedDateTime.of(2025, 10, 26, 8, 0, 0, 0, ZoneId.of("Europe/Berlin"))
    )
)